# Reproduce Berkeley's Official APR — in your browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/DEMO_apr_vs_hcd_colab.ipynb)

**What this is.** A fully reproducible audit: starting from published primary-source completion data,
this notebook **derives** Berkeley's housing certificate-of-occupancy counts for CY2024 and CY2025 —
**CY2024 = 709**, **CY2025 = 532** net-new units (private housing; UC student housing excluded as group
quarters per HCD rules) — and compares them to the city's own submitted figures from the state CKAN portal.

Nothing is pre-computed here: the numbers are calculated live from the rows, in front of you. `Runtime -> Run all`.

### What's shared, and why
Only **two small derived CSVs** are published (a few hundred rows each):
- `apr_completions_extract.csv` — one row per completed project (address, APN, CO date, net units, UC flag).
- `hcd_apr_comparison.csv` — the city's CO rows from CKAN (to reproduce the de-duplication live).

The **full canonical pipeline database stays private** (gitignored). These extracts contain exactly the
fields needed to reproduce the APR completion counts and nothing more — so the result is auditable without
exposing the whole working store. Provenance: derived from `berkeley_housing_v2.db`, SHA-256 `179434a8…`.

## Step 0 — Fetch the published data (no install, no local files)

In [ ]:
import pandas as pd, re

BASE_URL = 'https://raw.githubusercontent.com/blockXblock/berkeley-housing-analysis/main/data/public/'
completions = pd.read_csv(BASE_URL + 'apr_completions_extract.csv')
hcd         = pd.read_csv(BASE_URL + 'hcd_apr_comparison.csv')
completions['co_year'] = completions['co_year'].astype(str)

print(f'completions extract : {len(completions)} rows  (all CO-issued projects)')
print(f'HCD/CKAN comparison : {len(hcd)} rows  (city CO rows, 2024-2025)')
completions.head(4)

## Step 1 — Derive our APR completion counts

Sum net-new units by the year the certificate of occupancy was issued, then apply the **group-quarters
exclusion** HCD requires (UC Berkeley dormitories cannot count as HCD units). The notebook shows *both*
numbers so you can see why the filter matters: without it, CY2024 reads **1,009** (the 4 UC projects,
+300 from 1950 Oxford St, inflate it); with it, **709**.

In [ ]:
is_uc = completions['is_uc_project'].astype(str).str.lower().isin(['true', '1'])

incl_uc = completions.groupby('co_year')['net_units'].sum()                 # before GQ filter
private = completions[~is_uc]                                               # HCD group-quarters exclusion
ours    = private.groupby('co_year')['net_units'].sum()

OURS = {y: int(ours.get(y, 0)) for y in ('2024', '2025', '2026')}
for y in ('2024', '2025', '2026'):
    extra = f'    [without GQ filter: {int(incl_uc.get(y,0))}]' if y == '2024' else ''
    print(f'CY{y}:  {OURS[y]:>4} net-new CO units (private, UC-excluded){extra}')

## Step 2 — The city's official figures (CKAN), with the de-duplication reproduced live

The city's submitted CY2025 rows contain within-year duplicates (the same parcel on several rows). We
collapse rows sharing a **parcel key** (APN, else normalized street address; repeats carry identical units →
keep one per key). CY2025 raw **984 → 487** deduped.

In [ ]:
def parcel_key(apn, addr):
    a = re.sub(r'[^\d]', '', str(apn) if pd.notna(apn) else '')
    if a:
        return a
    s = str(addr or '').upper().split(',')[0]
    m = re.match(r'\s*(\d+)', s)
    if not m:
        return 'X'
    rest = re.sub(r'^\s*\d+(-\d+)?\s+', '', s)
    w = re.sub(r'[^A-Z ]', '', rest).split()
    return 'X' + m.group(1) + '|' + (w[0] if w else '')

hcd['key'] = [parcel_key(a, s) for a, s in zip(hcd['apn'], hcd['street_address'])]
raw    = hcd.groupby('year')['co_units'].sum()
dedup  = hcd.groupby(['year', 'key'])['co_units'].max().groupby('year').sum()
HCD = {int(y): int(dedup[y]) for y in dedup.index}
for y in (2024, 2025):
    note = '' if int(raw[y]) == HCD[y] else f'   (raw {int(raw[y])} -> {HCD[y]} after parcel-key dedup)'
    print(f'CY{y}:  HCD/CKAN deduped CO units = {HCD[y]}{note}')

## Step 3 — Side-by-side: our reproduction vs the city's submitted APR

In [ ]:
def explain(year, ours_n, hcd_n):
    if year == 2024:
        return f'match — comprehensive primary-source count equals the city ({ours_n} vs {hcd_n})'
    return (f'+{ours_n-hcd_n}: comprehensive ADU coverage + units we capture complete that CKAN still '
            f'shows permit-only (e.g. 1367 University). City 984 raw -> {hcd_n} deduped.')

comparison = pd.DataFrame([
    {'Year': f'CY{y}', 'Our APR (UC-excluded)': OURS[str(y)], 'HCD / CKAN (deduped)': HCD[y],
     'Delta': OURS[str(y)] - HCD[y], 'Explanation': explain(y, OURS[str(y)], HCD[y])}
    for y in (2024, 2025)
])
comparison

## Correctness gate

This notebook must **derive** exactly 709 (CY2024) and 532 (CY2025) — the figures in the project's audit
record and on the live dashboard (berkeleybuild.com). Anything else (1,009 = group-quarters filter dropped;
786 = stale data) is wrong, and this cell fails loudly.

In [ ]:
EXPECTED = {'2024': 709, '2025': 532}
for y, exp in EXPECTED.items():
    got = OURS[y]
    print(f"[{'PASS' if got == exp else 'FAIL'}] CY{y}: derived {got}, expected {exp}")
    assert got == exp, f'CY{y} derived {got}, expected {exp} — reproduction is WRONG'
print('\nAll checks PASS - derived live from the published rows, matching the audit record and dashboard.')